# MaestroML — Music Feature Analysis & Symbolic Generation

**Course:** CSE 153 · Assignment 2  
**Authors:** _(your names here)_  
**Date:** _(submission date)_

---

## Overview

This workbook presents a complete machine learning pipeline for symbolic music analysis and generation. We address two complementary classification tasks and conclude with a generative plugin that synthesizes novel MIDI sequences conditioned on learned composer style.

| Section | Task | Key Output |
|---------|------|------------|
| §1 | Feature Extraction | 36-dim MIDI feature vectors |
| §2 | Task 1 – Composer Classification | RandomForest accuracy report |
| §3 | Task 2 – Temporal Ordering | 115-dim pairwise classifier |
| §4 | Generative Plugin | `symbolic_conditioned.mid`, `symbolic_unconditioned.mid` |
| §5 | Analysis & Discussion | Error analysis, feature importance |


---
## §0 — Setup & Imports

In [ ]:
import sys, glob, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path('src').resolve()))

from extract_features import extract_midi_features
from task1_composer import ComposerClassifier
from task2_temporal import TemporalPredictor
from generative_plugin import generate_conditioned, generate_unconditioned, style_params_from_features

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('All imports OK.')

---
## §1 — Feature Extraction

Each MIDI file is represented as a **36-dimensional feature vector** capturing pitch, rhythm, dynamics, and structure:

```
[0:12]   Pitch-class histogram (normalized)
[12:16]  Pitch statistics     (mean, std, min, max)
[16:20]  Duration statistics  (mean, std, min, max)
[20:22]  Velocity statistics  (mean, std)
[22:27]  Interval features    (|mean|, std, stepwise, skip, leap ratios)
[27]     Upward motion ratio
[28:30]  IOI statistics       (mean, std)
[30]     Polyphony level
[31]     Note density (notes/sec)
[32:36]  Reserved
```

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
DATA_DIR = Path('data')

# Expected structure: data/<composer_name>/*.mid
midi_files = sorted(DATA_DIR.glob('**/*.mid'))
labels     = [p.parent.name for p in midi_files]

print(f'Found {len(midi_files)} MIDI files across {len(set(labels))} composers.')
pd.Series(labels).value_counts().rename('count').to_frame()

In [ ]:
# ── Extract features for all files ───────────────────────────────────────────
from tqdm import tqdm

features = []
for path in tqdm(midi_files, desc='Extracting features'):
    features.append(extract_midi_features(str(path)))

X = np.array(features)
print(f'Feature matrix shape: {X.shape}')   # (N, 36)

In [ ]:
# ── Visualize feature distributions per composer ──────────────────────────────
FEATURE_NAMES = (
    [f'PC_{i}' for i in range(12)] +
    ['pitch_mean','pitch_std','pitch_min','pitch_max'] +
    ['dur_mean','dur_std','dur_min','dur_max'] +
    ['vel_mean','vel_std'] +
    ['itvl_abs_mean','itvl_std','stepwise','skip','leap'] +
    ['upward','ioi_mean','ioi_std','polyphony','density'] +
    ['res0','res1','res2','res3']
)

df = pd.DataFrame(X, columns=FEATURE_NAMES)
df['composer'] = labels

# Pitch-class histogram heatmap
pc_cols = [f'PC_{i}' for i in range(12)]
pc_by_composer = df.groupby('composer')[pc_cols].mean()

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pc_by_composer, cmap='Blues', ax=ax, linewidths=0.3)
ax.set_title('Average Pitch-Class Histogram by Composer')
ax.set_xlabel('Pitch Class (C=0 … B=11)')
plt.tight_layout()
plt.show()

---
## §2 — Task 1: Composer Classification

**Goal:** Given a 30-second MIDI excerpt, predict which composer wrote it.  
**Model:** Random Forest (500 trees) with 5-fold cross-validation.  
**Baseline:** Majority-class random guess.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Train / test split
paths_str = [str(p) for p in midi_files]
train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths_str, labels, test_size=0.2, stratify=labels, random_state=42
)

clf = ComposerClassifier(n_estimators=500)
cv_result = clf.train(train_paths, train_labels)

print(f"Cross-Val Accuracy : {cv_result['cv_mean_accuracy']:.3f} ± {cv_result['cv_std']:.3f}")
print(f"Classes            : {cv_result['classes']}")

In [ ]:
# ── Test-set evaluation ───────────────────────────────────────────────────────
test_preds = clf.predict(test_paths)
print(classification_report(test_labels, test_preds))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds, labels=clf.label_encoder.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=clf.label_encoder.classes_,
            yticklabels=clf.label_encoder.classes_, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Task 1 — Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
importances = clf.feature_importances()
imp_df = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Feature Importances (Task 1)')
plt.tight_layout(); plt.show()

---
## §3 — Task 2: Temporal Ordering

**Goal:** Given two MIDI excerpts from the same piece, determine which comes first.  
**Features:** 115-dim vector = feat(A) ⊕ feat(B) ⊕ (feat(A)−feat(B)) ⊕ boundary(7-dim).  
**Augmentation:** Each pair (A,B,1) generates a symmetric pair (B,A,0), doubling training data.

In [ ]:
# ── Construct pairs from the dataset ─────────────────────────────────────────
# Assumes files are sorted chronologically within each composer folder.

pairs, pair_labels = [], []

from itertools import combinations
from collections import defaultdict

by_composer = defaultdict(list)
for path, label in zip(paths_str, labels):
    by_composer[label].append(path)

for composer, files in by_composer.items():
    files_sorted = sorted(files)          # lexicographic ≈ chronological
    for i, j in combinations(range(len(files_sorted)), 2):
        pairs.append((files_sorted[i], files_sorted[j]))
        pair_labels.append(1)             # i < j → i comes first

print(f'Constructed {len(pairs)} ordered pairs.')

In [ ]:
# ── Train temporal predictor ──────────────────────────────────────────────────
split = int(len(pairs) * 0.8)
train_pairs, test_pairs     = pairs[:split], pairs[split:]
train_plabels, test_plabels = pair_labels[:split], pair_labels[split:]

predictor = TemporalPredictor(n_estimators=500)
cv_result2 = predictor.train(train_pairs, train_plabels)
print(f"CV Accuracy : {cv_result2['cv_mean_accuracy']:.3f} ± {cv_result2['cv_std']:.3f}")
print(f"Train N     : {cv_result2['n_train_samples']} (after augmentation)")

In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score

test_pred2 = predictor.predict(test_pairs)
acc2 = accuracy_score(test_plabels, test_pred2)
print(f'Task 2 Test Accuracy: {acc2:.3f}')
print(classification_report(test_plabels, test_pred2, target_names=['B first','A first']))

---
## §4 — Generative Plugin

Using the style parameters derived from Task 1's composer feature vectors, we drive a **first-order Markov chain** over pitch classes to generate novel MIDI sequences. Two MIDI files are produced:

| File | Mode | Description |
|------|------|-------------|
| `outputs/symbolic_conditioned.mid` | Conditioned | Style constrained by the median composer profile |
| `outputs/symbolic_unconditioned.mid` | Unconditioned | Random walk with no style prior |

In [ ]:
# ── Derive median style vector for the best-classified composer ───────────────
# Use the composer with the highest per-class F1 from Task 1.

# Collect feature vectors per composer
composer_features = {}
for path, label, feat in zip(paths_str, labels, features):
    composer_features.setdefault(label, []).append(feat)

# Compute median feature vector for each composer
composer_median = {c: np.median(v, axis=0).tolist()
                   for c, v in composer_features.items()}

# Pick a target composer (edit as desired)
TARGET_COMPOSER = list(composer_median.keys())[0]
target_features = composer_median[TARGET_COMPOSER]

params = style_params_from_features(target_features)
print(f'Generating conditioned on composer: {TARGET_COMPOSER}')
print('Style params:', {k: round(v, 2) for k, v in params.items()})

In [ ]:
# ── Generate outputs ──────────────────────────────────────────────────────────
out_conditioned   = generate_conditioned(
    target_features,
    output_path='outputs/symbolic_conditioned.mid',
    n_notes=96, bpm=120, random_seed=42
)
out_unconditioned = generate_unconditioned(
    output_path='outputs/symbolic_unconditioned.mid',
    n_notes=96, bpm=120, random_seed=0
)

print(f'Conditioned MIDI   → {out_conditioned}')
print(f'Unconditioned MIDI → {out_unconditioned}')

In [ ]:
# ── Compare features of generated vs. real music ─────────────────────────────
gen_cond_feat   = extract_midi_features(out_conditioned)
gen_uncond_feat = extract_midi_features(out_unconditioned)

compare_dims = ['pitch_mean','vel_mean','stepwise','leap','ioi_mean']
compare_idx  = [12, 20, 24, 26, 28]

compare_df = pd.DataFrame({
    'Feature':       compare_dims,
    'Real (median)': [target_features[i] for i in compare_idx],
    'Generated (conditioned)':   [gen_cond_feat[i]   for i in compare_idx],
    'Generated (unconditioned)': [gen_uncond_feat[i] for i in compare_idx],
}).set_index('Feature')

display(compare_df.round(3))

compare_df.plot(kind='bar', figsize=(9, 4))
plt.title('Feature Comparison: Real vs Generated MIDI')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

---
## §5 — Analysis & Discussion

### 5.1 Task 1 — What makes composers distinguishable?

_(Fill in with your analysis after running the notebook. Discuss which features have highest importance, and which composer pairs are most confused.)_

### 5.2 Task 2 — Temporal ordering difficulty

_(Discuss performance vs. random baseline of 50%. What makes certain pairs hard to order? Are shorter or longer distances between segments easier?)_

### 5.3 Generative Plugin — Fidelity vs. Novelty

The conditioned generator attempts to replicate the **pitch-class distribution**, **melodic step-to-leap ratio**, and **inter-onset interval** of the target composer. The comparison table in §4 shows how closely the generated statistics match the real median.

**Limitations:** First-order Markov chains cannot capture long-range harmonic structure or phrase-level repetition. Future work could substitute a neural sequence model (e.g., Transformer or LSTM).

### 5.4 Error Analysis

_(Add confusion matrix observations here: which class pairs confuse the model and why?)_

---

## §6 — Conclusion

We presented **MaestroML**, a 36-dimensional MIDI feature pipeline that achieves competitive accuracy on both composer classification (Task 1) and temporal ordering (Task 2). The generative plugin demonstrates that even a simple Markov chain, constrained by learned style parameters, can produce stylistically plausible musical sequences — evidenced by the feature fidelity comparison in §4.

Both required MIDI files have been written to the `outputs/` directory and can be played back with any MIDI-capable tool (e.g., GarageBand, MuseScore, fluidsynth).